# Gemma Grounded Health v3b: GRPO

Thin runner (D-10): repo at `7f1273c6f441`, recipe `configs/recipe_v3b_grpo.yaml`, main run, 300 steps, dev scoring. v3b continues training v2's adapter with a reward checked by two local models (no judge, no API call). All logic lives in the tested `ggh` package; any failing step stops the run.

In [ ]:
import os, subprocess, shutil, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # one T4 (D-9), even on a T4 x2 machine
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def sh(cmd):
    """Run a shell command; a non-zero exit stops the notebook."""
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

sh("git clone -q https://github.com/biplob819/gemma-grounded-health /kaggle/working/ggh")
os.chdir("/kaggle/working/ggh")
sh("git checkout -q 7f1273c6f4418ab745b79e694515ad990e747a4f")
sh("pip install -q -r notebooks/requirements-kaggle-grpo.txt")
sh("pip install -q --no-deps -e .")
sh("pip install -q --no-deps 'evalkit-harness==0.4.0'")

In [ ]:
sh("pip list 2>/dev/null | grep -i -E '^(torch|transformers|trl|peft|accelerate|bitsandbytes|datasets|unsloth|unsloth.zoo|evalkit-harness) ' || true")
sh("ggh doctor --kaggle --work /kaggle/working")

In [ ]:
# rebuild N1.2 data (the dev prompts); every committed hash is checked
sh("ggh data")
sh("git diff --exit-code docs/measurements/data_manifest.json")

In [ ]:
# the train kernel's run folder: the one holding report.json and final/
import glob
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/report.json", recursive=True)
         if os.path.isdir(os.path.join(os.path.dirname(p), "final"))]
assert len(found) == 1, f"want one GRPO run under /kaggle/input, got {found}"
run = found[0]
print(run, sorted(os.listdir(run)))

In [ ]:
# v2 and each checkpoint, greedy, on the first dev prompts, scored by the reward
sh(f"ggh rl-dev-b --config configs/recipe_v3b_grpo.yaml --run {run} --out /kaggle/working/dev --n 300")

In [ ]:
shutil.copy("/kaggle/working/dev/rl_dev.json", "/kaggle/working/rl_dev.json")
print(open("/kaggle/working/dev/rl_dev.json").read())